In [26]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm
from matplotlib.gridspec import GridSpec
from scipy.stats import wilcoxon
import os
import warnings

In [28]:
# ---- Config ----
CSV_PATH = "../outputs/microexon_final.csv"
OUT_DIR = "../outputs/"
FONT_SIZE = 18

# ---- Colors ----
# Violin fills: grey (rest) + orange (microexon)
# Connecting line colormap: grey -> white -> orange
cmap = LinearSegmentedColormap.from_list(
    "micro_rest", ["#555555", "#FFFFFF", "#D55E00"]
)

# ---- Fonts: Arial, uniform size ----
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "figure.dpi": 300, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "svg.fonttype": "none",
})

# ---- Load data ----
df = pd.read_csv(CSV_PATH)

props = [
    ("Aromaticity",          "Aromaticity_miniexon",    "Aromaticity_rest"),
    ("Hydrophobicity ratio", "HydrophRatio_miniexon",   "HydrophRatio_rest"),
    ("Charge density",       "ChargeDensity_miniexon",  "ChargeDensity_rest"),
    ("Disorder score",       "exon_disorder_score",     "protein_disorder_score"),
]

# ---- Helper: extract seaborn KDE curve (data on x, density on y) ----
def get_sns_kde(data):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _fig, _ax = plt.subplots(figsize=(1, 1))
        sns.kdeplot(data, ax=_ax)
        line = _ax.lines[-1]
        x, y = line.get_xdata(), line.get_ydata()
        plt.close(_fig)
    return x, y

# ---- Figure: 2x2 grid + dedicated colorbar column ----
fig = plt.figure(figsize=(12, 9), facecolor="white")
gs = GridSpec(2, 3, width_ratios=[1, 1, 0.06], wspace=0.35, hspace=0.35,
              left=0.08, right=0.96, top=0.96, bottom=0.08)
axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]

y_rest = 0.0   # bottom half-violin (faces down)
y_micro = 0.3  # top half-violin (faces up)
violin_half_height = 0.12

for i, (name, mc, rc) in enumerate(props):
    ax = axes[i]
    mini = df[mc].dropna().values
    rest = df[rc].dropna().values
    n = min(len(mini), len(rest))
    mini, rest = mini[:n], rest[:n]
    diff = mini - rest

    # Per-panel normalization
    dmin, dmax = diff.min(), diff.max()
    panel_norm = TwoSlopeNorm(vcenter=0, vmin=dmin, vmax=dmax)
    order = np.argsort(np.abs(diff))

    # --- Extract KDE curves from seaborn (x=data, y=density) ---
    x_w, y_w = get_sns_kde(rest)
    x_m, y_m = get_sns_kde(mini)

    # Shared x-grid spanning both distributions
    x_min = min(x_w.min(), x_m.min())
    x_max = max(x_w.max(), x_m.max())
    x_grid = np.linspace(x_min, x_max, 500)

    # Interpolate densities onto shared grid
    density_w = np.interp(x_grid, x_w, y_w, left=0, right=0)
    density_m = np.interp(x_grid, x_m, y_m, left=0, right=0)

    # Scale each independently to same peak height
    scale_w = violin_half_height / density_w.max()
    scale_m = violin_half_height / density_m.max()

    # --- Half-violin: Rest (bottom, faces down) ---
    ax.fill_between(x_grid, y_rest, y_rest - density_w * scale_w,
                     color="#D9D9D9", edgecolor="black", linewidth=0.8, zorder=3)
    ax.plot(x_grid, y_rest - density_w * scale_w,
            color="black", linewidth=0.8, zorder=4)

    # --- Half-violin: Microexon (top, faces up) ---
    ax.fill_between(x_grid, y_micro, y_micro + density_m * scale_m,
                     color="#FDD9A0", edgecolor="#D55E00", linewidth=0.8, zorder=3)
    ax.plot(x_grid, y_micro + density_m * scale_m,
            color="#D55E00", linewidth=0.8, zorder=4)

    # --- Paired connecting lines ---
    for rank, idx in enumerate(order):
        color = cmap(panel_norm(diff[idx]))
        ax.plot([rest[idx], mini[idx]], [y_rest, y_micro],
                color=color, linewidth=1.0, alpha=0.5, zorder=2 + rank * 0.01)
        ax.scatter([rest[idx], mini[idx]], [y_rest, y_micro],
                   s=16, color=color, edgecolor="white", linewidth=0.3,
                   zorder=10 + rank * 0.01)

    # --- Axes ---
    ax.set_yticks([y_rest, y_micro])
    ax.set_yticklabels(["Rest of\nProtein", "Microexon"])
    ax.set_xlabel(name)
    ax.set_ylim(-0.2, 0.55)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=FONT_SIZE, length=4, width=0.8)

# ---- Colorbar in dedicated column ----
cax = fig.add_subplot(gs[:, 2])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=Normalize(vmin=-1, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Microexon > Rest  ←  Difference  →  Microexon < Rest",
               fontsize=FONT_SIZE, labelpad=10)
cbar.set_ticks([])
cbar.outline.set_linewidth(0.8)

path_png = os.path.join(OUT_DIR, "microexon_paired_halfviolin.png")
path_svg = os.path.join(OUT_DIR, "microexon_paired_halfviolin.svg")
fig.savefig(path_png, bbox_inches="tight", facecolor="white")
fig.savefig(path_svg, bbox_inches="tight", facecolor="white")
plt.close()
print(f"Saved: {path_png}")
print(f"Saved: {path_svg}")

Saved: ../outputs/microexon_paired_halfviolin.png
Saved: ../outputs/microexon_paired_halfviolin.svg
